In [52]:
import glob
import os
import random
import numpy as np
import csv
import json

In [53]:
class MajorityClassifier:
    def fit(self, X, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        self.most_frequent = max(counts, key=counts.get)
        return self
 
    def predict(self, X):
        return [self.most_frequent for _ in X]   

In [55]:
def train_test_split(X, y, test_size=0.2, seed=42):
    idx = list(range(len(y)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(y) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

In [56]:
def get_accuracy(y_true, y_pred):
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

In [57]:
DATA_ROOT = "/path/to/autoPET"
 
# Build labels, AI helped with this snippet
X, y = [], []
for path in sorted(glob.glob(os.path.join(DATA_ROOT, "labelsTr", "*.nii.gz"))):
    mask = nib.load(path).dataobj
    X.append(os.path.basename(path))
    y.append(1 if mask[...].any() else 0)
 
print(f"Scans: {len(y)}  |  tumor: {sum(y)}  |  no tumor: {len(y) - sum(y)}") # AI for print

Scans: 0  |  tumor: 0  |  no tumor: 0


In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
#clf = MajorityClassifier().fit(X_train, y_train)
#print(f"Most frequent class: {clf.most_frequent}") # AI for print
#y_pred = clf.predict(X_test)
#print(f"Baseline Accuracy: {get_accuracy(y_test, y_pred):.4f}") # AI for print

In [59]:
# Load FDG labels (AI helped with this)
labels = {}
with open("data/fdg_metadata.csv") as f:
    for row in csv.DictReader(f):
        if row["Modality"] == "PT":
            patient = row["Subject ID"].replace("PETCT_", "")
            study = row["File Location"].split("/")[-2]
            name = f"fdg_{patient}_{study}"
            labels[name] = 0 if row["diagnosis"] == "NEGATIVE" else 1

# Load the first training/validation split
with open("data/splits_final.json") as f:
    split = json.load(f)[0]

X_train = [x for x in split["train"] if x in labels]
X_test = [x for x in split["val"] if x in labels]

y_train = [labels[x] for x in X_train]
y_test = [labels[x] for x in X_test]

# Test existing classifier
model = MajorityClassifier().fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training scans:", len(X_train))
print("Testing scans:", len(X_test))
print("Accuracy:", get_accuracy(y_test, predictions))

Training scans: 822
Testing scans: 192
Accuracy: 0.4375


In [187]:
#Binary classification components. no encoder or pretrained weights included

import torch
from torch import nn


class BinaryClassificationHead(nn.Module):
    #Map [B, C, D, H, W] features (or pooled [B, C]) to [B, 1] logits.

    def __init__(self, in_features: int):
        super().__init__()
        if in_features <= 0:
            raise ValueError("in_features must be positive")
        self.network = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if not isinstance(features, torch.Tensor):
            raise TypeError("Expected a feature tensor")

        if features.ndim != 2:
            raise ValueError("Expected [B, C] features")

        if features.shape[1] != 640:
            raise ValueError("Expected 640 input features")

        return self.network(features)

In [188]:
def train_classifier(X_train, y_train, epochs=2000):
    model = BinaryClassificationHead(in_features=640)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # X_train must contain encoder feature tensors of shape [640]
    features = torch.tensor(X_train, dtype=torch.float32)
    labels = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

    return model

In [191]:
import numpy as np
data = np.load("features/encoder_features.npz")
X = data['features']
Y = data['lesion']
split = data['split']
X_train = X[split == "train"]
X_test = X[split == "test"]
X_val = X[split == "val"]
Y_train = Y[split == "train"]
Y_test = Y[split == "test"]
Y_val = Y[split == "val"]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



In [194]:
model = train_classifier(X_train_scaled, Y_train)

Epoch 1000/2000, Loss: 0.0097
Epoch 2000/2000, Loss: 0.0033


In [195]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
model.eval()
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    logits = model(X_val_tensor)
    probability = torch.sigmoid(logits)
    predictions = (probability >= 0.5).int().numpy().flatten()

print("Validation Accuracy:", accuracy_score(Y_val, predictions))
print(classification_report(Y_val, predictions))
print("Confusion Matrix:")
print(confusion_matrix(Y_val, predictions))

Validation Accuracy: 0.726984126984127
              precision    recall  f1-score   support

           0       0.58      0.59      0.59       103
           1       0.80      0.79      0.80       212

    accuracy                           0.73       315
   macro avg       0.69      0.69      0.69       315
weighted avg       0.73      0.73      0.73       315

Confusion Matrix:
[[ 61  42]
 [ 44 168]]


## Retraining with epoch-760 features

The cells below compare the new features and save the validation-selected model.


In [ ]:
#Binary classification components. no encoder or pretrained weights included

import torch
from torch import nn


class Ep760ClassificationHead(nn.Module):

    def __init__(self, in_features: int, hidden_size=128, dropout=0.5):
        super().__init__()
        if in_features <= 0:
            raise ValueError("in_features must be positive")
        self.network = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if not isinstance(features, torch.Tensor):
            raise TypeError("Expected a feature tensor")

        if features.ndim != 2:
            raise ValueError("Expected [B, C] features")

        if features.shape[1] != self.network[0].in_features:
            raise ValueError("Feature count does not match the model")

        return self.network(features)

In [ ]:
from copy import deepcopy


def train_ep760_classifier(X_train, y_train, X_val, y_val, epochs=2000,
                     hidden_size=128, dropout=0.5, lr=5e-4, weight_decay=1e-4):
    torch.manual_seed(42)
    model = Ep760ClassificationHead(X_train.shape[1], hidden_size, dropout)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    features = torch.as_tensor(X_train, dtype=torch.float32)
    labels = torch.as_tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    val_features = torch.as_tensor(X_val, dtype=torch.float32)
    val_labels = torch.as_tensor(y_val, dtype=torch.float32).reshape(-1, 1)
    best_loss = float("inf")
    best_epoch = 0
    best_weights = None
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(val_features), val_labels)
        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss.item())
        if val_loss.item() < best_loss:
            best_loss = val_loss.item()
            best_epoch = epoch + 1
            # AI helped copy the best weights so later epochs do not change them.
            best_weights = deepcopy(model.state_dict())
        if epoch == 0 or (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch + 1}: train {loss.item():.4f}, val {val_loss.item():.4f}")

    model.load_state_dict(best_weights)
    model.eval()
    print(f"Best epoch: {best_epoch}, validation loss: {best_loss:.4f}")
    return model, history, best_epoch


In [ ]:
import hashlib
from pathlib import Path
import numpy as np

feature_path = Path("features/encoder_features_ep760.npz")
with np.load(feature_path, allow_pickle=False) as data:
    feature_split = data["split"]
    train_mask = feature_split == "train"
    val_mask = feature_split == "val"
    X_train = data["features"][train_mask].copy()
    X_val = data["features"][val_mask].copy()
    Y_train = data["lesion"][train_mask].copy()
    Y_val = data["lesion"][val_mask].copy()
    val_tracers = data["tracer"][val_mask].copy()
    train_ids = data["case_ids"][train_mask].copy()
    val_ids = data["case_ids"][val_mask].copy()
    encoder_metadata = json.loads(data["metadata"].item())

assert X_train.shape == (980, 640) and X_val.shape == (315, 640)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

# AI helped check that patients are not shared between the two splits.
assert not ({"_".join(c.split("_")[:2]) for c in train_ids}
            & {"_".join(c.split("_")[:2]) for c in val_ids})

feature_mean = X_train.mean(axis=0, keepdims=True)
feature_std = X_train.std(axis=0, keepdims=True)
# AI helped avoid dividing by a near-zero standard deviation.
feature_std[feature_std < 1e-6] = 1
X_train_scaled = (X_train - feature_mean) / feature_std
X_val_scaled = (X_val - feature_mean) / feature_std
print("Training:", X_train.shape, "Validation:", X_val.shape)
print("Encoder checkpoint:", encoder_metadata["checkpoint_name"])


Training: (980, 640) Validation: (315, 640)
Encoder checkpoint: checkpoint_ep760.pth


In [ ]:
torch.set_num_threads(1)

print("Run: 128 hidden, dropout")
large_model, large_history, large_epoch = train_ep760_classifier(
    X_train_scaled, Y_train, X_val_scaled, Y_val
)

print("Run: 16 hidden, previous settings")
small_model, small_history, small_epoch = train_ep760_classifier(
    X_train_scaled, Y_train, X_val_scaled, Y_val,
    epochs=500, hidden_size=16, dropout=0.0, lr=0.001, weight_decay=0.01
)

model = large_model
history = large_history
best_epoch = large_epoch
model_config = {"name": "128 hidden, dropout", "hidden_size": 128,
                "dropout": 0.5, "lr": 5e-4, "weight_decay": 1e-4, "epochs": 2000}
# AI helped compare the two runs using their best validation loss.
if min(small_history["val_loss"]) < min(large_history["val_loss"]):
    model = small_model
    history = small_history
    best_epoch = small_epoch
    model_config = {"name": "16 hidden, previous settings", "hidden_size": 16,
                    "dropout": 0.0, "lr": 0.001, "weight_decay": 0.01, "epochs": 500}
print("Selected:", model_config["name"])


Run: 128 hidden, dropout
Epoch 1: train 0.7193, val 0.6574
Epoch 200: train 0.1356, val 0.3675
Epoch 400: train 0.0412, val 0.4449
Epoch 600: train 0.0167, val 0.5181
Epoch 800: train 0.0089, val 0.5787
Epoch 1000: train 0.0081, val 0.6302
Epoch 1200: train 0.0052, val 0.6437
Epoch 1400: train 0.0047, val 0.6678
Epoch 1600: train 0.0032, val 0.6838
Epoch 1800: train 0.0033, val 0.7056
Epoch 2000: train 0.0023, val 0.7174
Best epoch: 147, validation loss: 0.3605
Run: 16 hidden, previous settings
Epoch 1: train 0.6627, val 0.6306
Epoch 200: train 0.1725, val 0.3523
Epoch 400: train 0.1029, val 0.3628
Best epoch: 216, validation loss: 0.3518
Selected: 16 hidden, previous settings


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score


# AI helped add balanced accuracy, sensitivity, specificity, and AUC.
def metrics(actual, scores, threshold=0.5):
    predicted = scores >= threshold
    matrix = confusion_matrix(actual, predicted, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return {"accuracy": float(accuracy_score(actual, predicted)),
            "balanced_accuracy": float(balanced_accuracy_score(actual, predicted)),
            "sensitivity": float(tp / (tp + fn)), "specificity": float(tn / (tn + fp)),
            "auc": float(roc_auc_score(actual, scores)), "confusion_matrix": matrix.tolist()}


def show_metrics(name, result):
    print(name)
    print(f"Accuracy {result['accuracy']:.2%}, balanced accuracy {result['balanced_accuracy']:.2%}")
    print(f"Sensitivity {result['sensitivity']:.2%}, specificity {result['specificity']:.2%}, AUC {result['auc']:.4f}")
    print("Confusion matrix (actual rows, predicted columns; 0, 1):", result["confusion_matrix"])


with torch.no_grad():
    val_probabilities = model(torch.tensor(X_val_scaled)).sigmoid().flatten().numpy()
# AI helped choose a threshold using validation data only.
thresholds = (0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)
best_balanced_accuracy = -1
for threshold in thresholds:
    result = metrics(Y_val, val_probabilities, threshold)
    print(f"Threshold {threshold:.2f}: accuracy {result['accuracy']:.2%}, balanced {result['balanced_accuracy']:.2%}")
    if result["balanced_accuracy"] > best_balanced_accuracy:
        best_balanced_accuracy = result["balanced_accuracy"]
        selected_threshold = threshold
        validation_result = result
show_metrics("Epoch-760 features: selected threshold " + str(selected_threshold), validation_result)
for tracer in ("fdg", "psma"):
    mask = val_tracers == tracer
    show_metrics(tracer.upper(), metrics(Y_val[mask], val_probabilities[mask], selected_threshold))
majority = int(Y_train.mean() >= 0.5)
show_metrics("Training-majority baseline", metrics(Y_val, np.full(len(Y_val), majority)))

# AI helped compare v2 on the same validation scans using its original features.
old_checkpoint = torch.load("checkpoints/improved_v2.pt", map_location="cpu", weights_only=True)
with np.load("features/encoder_features.npz", allow_pickle=False) as old:
    old_mask = old["split"] == "val"
    assert np.array_equal(old["case_ids"][old_mask], val_ids)
    assert np.array_equal(old["lesion"][old_mask], Y_val)
    old_inputs = torch.tensor(old["features"][old_mask])
old_model = nn.Sequential(nn.Linear(640, 16), nn.ReLU(), nn.Linear(16, 1))
old_model.load_state_dict(old_checkpoint["model_state_dict"])
old_model.eval()
with torch.no_grad():
    old_scores = old_model((old_inputs - old_checkpoint["feature_mean"])
                          / old_checkpoint["feature_std"]).sigmoid().flatten().numpy()
show_metrics("Saved v2, original features", metrics(Y_val, old_scores, old_checkpoint["threshold"]))


Threshold 0.40: accuracy 84.13%, balanced 80.47%
Threshold 0.45: accuracy 85.40%, balanced 82.41%
Threshold 0.50: accuracy 86.03%, balanced 84.38%
Threshold 0.55: accuracy 85.71%, balanced 84.89%
Threshold 0.60: accuracy 85.40%, balanced 84.91%
Threshold 0.65: accuracy 82.54%, balanced 82.79%
Threshold 0.70: accuracy 80.32%, balanced 81.63%
Epoch-760 features: selected threshold 0.6
Accuracy 85.40%, balanced accuracy 84.91%
Sensitivity 86.32%, specificity 83.50%, AUC 0.9116
Confusion matrix (actual rows, predicted columns; 0, 1): [[86, 17], [29, 183]]
FDG
Accuracy 83.42%, balanced accuracy 83.64%
Sensitivity 80.19%, specificity 87.10%, AUC 0.9101
Confusion matrix (actual rows, predicted columns; 0, 1): [[81, 12], [21, 85]]
PSMA
Accuracy 88.79%, balanced accuracy 71.23%
Sensitivity 92.45%, specificity 50.00%, AUC 0.7245
Confusion matrix (actual rows, predicted columns; 0, 1): [[5, 5], [8, 98]]
Training-majority baseline
Accuracy 67.30%, balanced accuracy 50.00%
Sensitivity 100.00%, spec

In [ ]:
checkpoint_path = Path("checkpoints/encoder_ep760_v3.pt")
if checkpoint_path.exists():
    raise FileExistsError("Use a new checkpoint filename instead of overwriting this run")
# AI helped save the weights, scaling, settings, and feature-file hash together.
checkpoint = {
    "model_state_dict": model.state_dict(),
    "feature_mean": torch.tensor(feature_mean), "feature_std": torch.tensor(feature_std),
    "in_features": int(X_train.shape[1]), "threshold": selected_threshold,
    "config": model_config, "selected_epoch": best_epoch, "seed": 42,
    "architecture": "Ep760ClassificationHead", "validation_metrics": validation_result,
    "history": history, "encoder_metadata": encoder_metadata,
    "features_sha256": hashlib.sha256(feature_path.read_bytes()).hexdigest(),
    "torch_version": str(torch.__version__),
}
torch.save(checkpoint, checkpoint_path)
# AI helped reload the checkpoint and check that the predictions match.
loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = Ep760ClassificationHead(loaded["in_features"], loaded["config"]["hidden_size"], loaded["config"]["dropout"])
restored.load_state_dict(loaded["model_state_dict"])
restored.eval()
with torch.no_grad():
    restored_scores = restored((torch.tensor(X_val) - loaded["feature_mean"])
                               / loaded["feature_std"]).sigmoid().flatten()
    torch.testing.assert_close(restored_scores, torch.tensor(val_probabilities))
print("Saved and verified:", checkpoint_path)
print("No test predictions computed. Website unchanged.")


Saved and verified: checkpoints\encoder_ep760_v3.pt
No test predictions computed. Website unchanged.


## Epoch-760 results

Trained on 980 scans and validated on 315. The 16-neuron model had lower validation loss. Used its weights from epoch 216 with a threshold of 0.60.

| Validation metric | Previous model | New model |
|---|---:|---:|
| Accuracy | 72.06% | 85.40% |
| Balanced accuracy | 71.51% | 84.91% |
